In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Introduction](introyt1_tutorial.html) \|\|
[Tensors](tensors_deeper_tutorial.html) \|\| **Autograd** \|\| [Building
Models](modelsyt_tutorial.html) \|\| [TensorBoard
Support](tensorboardyt_tutorial.html) \|\| [Training
Models](trainingyt.html) \|\| [Model Understanding](captumyt.html)

The Fundamentals of Autograd
============================

Follow along with the video below or on
[youtube](https://www.youtube.com/watch?v=M0fX15_-xrY).



In [ ]:
# Run this cell to load the video
from IPython.display import display, HTML
html_code = """
<div style="margin-top:10px; margin-bottom:10px;">
  <iframe width="560" height="315" src="https://www.youtube.com/embed/M0fX15_-xrY" frameborder="0" allow="accelerometer; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>
</div>
"""
display(HTML(html_code))



PyTorch's *Autograd* feature is part of what make PyTorch flexible and
fast for building machine learning projects. It allows for the rapid and
easy computation of multiple partial derivatives (also referred to as
*gradients)* over a complex computation. This operation is central to
backpropagation-based neural network learning.  
PyTorch 的 Autograd（自动微分）功能正是 PyTorch 在构建机器学习项目时兼具灵活性与高效性的关键所在。它支持针对复杂计算，快速且便捷地计算多个偏导数（即 梯度）。这一操作是基于反向传播算法的神经网络学习的核心机制。

The power of autograd comes from the fact that it traces your
computation dynamically *at runtime,* meaning that if your model has
decision branches, or loops whose lengths are not known until runtime,
the computation will still be traced correctly, and you'll get correct
gradients to drive learning. This, combined with the fact that your
models are built in Python, offers far more flexibility than frameworks
that rely on static analysis of a more rigidly-structured model for
computing gradients.  
Autograd 的强大之处在于它能够在 运行时（at runtime） 动态追踪你的计算过程。这意味着，即使你的模型包含条件分支，或者循环次数在运行时才能确定，计算过程依然能被准确追踪，从而为你提供正确的梯度以驱动模型学习。此外，由于模型完全使用 Python 构建，与那些依赖静态分析且模型结构更为固定的框架相比，这种动态追踪机制提供了极大的灵活性。

What Do We Need Autograd For?  
为什么我们需要 Autograd？
-----------------------------


A machine learning model is a *function*, with inputs and outputs. For
this discussion, we'll treat the inputs as an *i*-dimensional vector
$\vec{x}$, with elements $x_{i}$. We can then express the model, *M*, as
a vector-valued function of the input: $\vec{y} =
\vec{M}(\vec{x})$. (We treat the value of M's output as a vector because
in general, a model may have any number of outputs.)   
机器学习模型本质上是一个具有输入和输出的 *函数*。在本次讨论中，我们将输入视为一个 *i* 维向量 $\vec{x}$，其元素为 $x_{i}$。因此，我们可以将模型 *M* 表示为关于输入的向量值函数：$\vec{y} = \vec{M}(\vec{x})$。（我们将 M 的输出视为向量，是因为通常情况下，模型可能具有任意数量的输出。）

Since we'll mostly be discussing autograd in the context of training,
our output of interest will be the model's loss. The *loss function*
L($\vec{y}$) = L($\vec{M}$($\vec{x}$)) is a single-valued scalar
function of the model's output. This function expresses how far off our
model's prediction was from a particular input's *ideal* output. *Note:
After this point, we will often omit the vector sign where it should be
contextually clear - e.g.,* $y$ instead of $\vec y$.  
由于我们主要在训练的背景下讨论 autograd，我们关注的输出将是模型的损失（loss）。*损失函数* L($\vec{y}$) = L($\vec{M}$($\vec{x}$)) 是模型输出的单值标量函数。该函数量化了对于特定输入，模型预测值与 *理想* 输出之间的偏差程度。*注：在此之后，如果上下文清晰，我们通常会省略向量符号——例如使用 $y$ 代替 $\vec y$。*

In training a model, we want to minimize the loss. In the idealized case
of a perfect model, that means adjusting its learning weights - that is,
the adjustable parameters of the function - such that loss is zero for
all inputs. In the real world, it means an iterative process of nudging
the learning weights until we see that we get a tolerable loss for a
wide variety of inputs.  
在训练模型时，我们的目标是使损失最小化。在完美模型的理想情况下，这意味着调整其学习权重（即函数的可调参数），使得对于所有输入，损失都为零。而在现实世界中，这意味着我们需要通过迭代过程不断微调学习权重，直到模型在面对各种各样的输入时，都能将损失降低到可接受的范围内。

How do we decide how far and in which direction to nudge the weights? We
want to *minimize* the loss, which means making its first derivative
with respect to the input equal to 0:
$\frac{\partial L}{\partial x} = 0$.  
那么，我们如何决定权重调整的方向和幅度呢？我们希望 *最小化* 损失，这在数学上意味着使其对输入的偏导数等于 0：$\frac{\partial L}{\partial x} = 0$。

Recall, though, that the loss is not *directly* derived from the input,
but a function of the model's output (which is a function of the input
directly), $\frac{\partial L}{\partial x}$ =
$\frac{\partial {L({\vec y})}}{\partial x}$. By the chain rule of
differential calculus, we have
$\frac{\partial {L({\vec y})}}{\partial x}$ =
$\frac{\partial L}{\partial y}\frac{\partial y}{\partial x}$ =
$\frac{\partial L}{\partial y}\frac{\partial M(x)}{\partial x}$.  
然而需要注意的是，损失并非 *直接* 由输入决定，而是模型输出的函数（而模型输出才是直接由输入决定的函数），即 $\frac{\partial L}{\partial x} = \frac{\partial {L({\vec y})}}{\partial x}$。根据微积分中的链式法则，我们有 $\frac{\partial {L({\vec y})}}{\partial x} = \frac{\partial L}{\partial y}\frac{\partial y}{\partial x} = \frac{\partial L}{\partial y}\frac{\partial M(x)}{\partial x}$。

$\frac{\partial M(x)}{\partial x}$ is where things get complex. The
partial derivatives of the model's outputs with respect to its inputs,
if we were to expand the expression using the chain rule again, would
involve many local partial derivatives over every multiplied learning
weight, every activation function, and every other mathematical
transformation in the model. The full expression for each such partial
derivative is the sum of the products of the local gradient of *every
possible path* through the computation graph that ends with the variable
whose gradient we are trying to measure.  
$\frac{\partial M(x)}{\partial x}$ 正是复杂性的来源。如果我们再次使用链式法则展开该表达式，模型输出相对于输入的偏导数将涉及模型中每一个乘法权重、每一个激活函数以及其他所有数学变换的众多局部偏导数。每一个此类偏导数的完整表达式，实际上是计算图中 *所有可能路径* 的局部梯度乘积之和，这些路径最终都汇聚于我们要计算梯度的变量。

In particular, the gradients over the learning weights are of interest
to us - they tell us *what direction to change each weight* to get the
loss function closer to zero.  
具体而言，我们最关心的是学习权重的梯度——它们指明了为了 *使损失函数趋近于零，每个权重应当朝哪个方向进行调整*。

Since the number of such local derivatives (each corresponding to a
separate path through the model's computation graph) will tend to go up
exponentially with the depth of a neural network, so does the complexity
in computing them. This is where autograd comes in: It tracks the
history of every computation. Every computed tensor in your PyTorch
model carries a history of its input tensors and the function used to
create it. Combined with the fact that PyTorch functions meant to act on
tensors each have a built-in implementation for computing their own
derivatives, this greatly speeds the computation of the local
derivatives needed for learning.  
由于这类局部导数的数量（每个导数对应模型计算图中的一条独立路径）往往会随着神经网络深度的增加呈指数级增长，计算它们的复杂度也会随之急剧上升。这正是 autograd 发挥作用的地方：它会追踪每一次计算的历史记录。在 PyTorch 模型中，每一个被计算出的张量都保留了其输入张量以及用于创建它的函数的历史记录。再加上 PyTorch 中用于操作张量的函数都内置了自身导数的计算实现，这极大地加速了学习过程中所需的局部导数计算。

A Simple Example  
一个简单的示例
================

That was a lot of theory - but what does it look like to use autograd in
practice?  
上面讲了大量的理论——但在实际中使用 autograd 究竟是怎样的呢？

Let's start with a straightforward example. First, we'll do some imports
to let us graph our results:  
让我们从一个简单的例子开始。首先，我们需要导入一些库以便绘制结果图表：


In [ ]:
# %matplotlib inline

import torch

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import math

Next, we'll create an input tensor full of evenly spaced values on the
interval $[0, 2{\pi}]$, and specify `requires_grad=True`. (Like most
functions that create tensors, `torch.linspace()` accepts an optional
`requires_grad` option.) Setting this flag means that in every
computation that follows, autograd will be accumulating the history of
the computation in the output tensors of that computation.  
接下来，我们将创建一个输入张量，它包含区间 $[0, 2{\pi}]$ 上均匀分布的值，并指定 `requires_grad=True`。（与大多数创建张量的函数一样，`torch.linspace()` 也接受一个可选的 `requires_grad` 参数。）设置此标志意味着在随后的每一次计算中，autograd 都会将该计算的历史记录累积到该计算的输出张量中。

In [ ]:
a = torch.linspace(0.0, 2.0 * math.pi, steps=25, requires_grad=True) # x轴均匀分布
print(a)

Next, we'll perform a computation, and plot its output in terms of its
inputs:  
接下来，我们将执行一次计算，并绘制其输出相对于输入的图像：

In [ ]:
b = torch.sin(a)
plt.plot(a.detach(), b.detach())

# 没有detach报错 RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead. 

Let's have a closer look at the tensor `b`. When we print it, we see an
indicator that it is tracking its computation history:  
让我们仔细查看一下张量 b。当我们打印它时，可以看到一个指示符，表明它正在追踪其计算历史：

In [ ]:
print(b) # b是a的sin得到，追踪变化

This `grad_fn` gives us a hint that when we execute the backpropagation
step and compute gradients, we'll need to compute the derivative of
$\sin(x)$ for all this tensor's inputs.  
这个 `grad_fn` 提示我们，当执行反向传播步骤并计算梯度时，我们需要计算 $\sin(x)$ 对该张量所有输入的导数。

Let's perform some more computations:  
让我们继续执行一些计算：



In [ ]:
c = 2 * b
print(c) # 乘法

d = c + 1
print(d) # 加法

Finally, let's compute a single-element output. When you call
`.backward()` on a tensor with no arguments, it expects the calling
tensor to contain only a single element, as is the case when computing a
loss function.  
最后，让我们计算一个单元素输出。当你在不带参数的情况下对张量调用 `.backward()` 时，它要求该调用张量仅包含单个元素，这在计算损失函数时正是如此。


In [ ]:
out = d.sum()
# out = d
print(out)

Each `grad_fn` stored with our tensors allows you to walk the
computation all the way back to its inputs with its `next_functions`
property. We can see below that drilling down on this property on `d`
shows us the gradient functions for all the prior tensors. Note that
`a.grad_fn` is reported as `None`, indicating that this was an input to
the function with no history of its own.    
存储在我们张量中的每个 `grad_fn` 都允许你通过其 `next_functions` 属性，沿着计算图一直回溯到最初的输入。我们在下图中可以看到，深入查看 `d` 的这一属性，可以展示出所有前置张量对应的梯度函数。需要注意的是，`a.grad_fn` 显示为 `None`，这表明它是一个函数的输入，且本身没有计算历史。

In [ ]:
print("d:")
print(d.grad_fn)
print(d.grad_fn.next_functions)
print(d.grad_fn.next_functions[0][0].next_functions)
print(d.grad_fn.next_functions[0][0].next_functions[0][0].next_functions)
print(d.grad_fn.next_functions[0][0].next_functions[0][0].next_functions[0][0].next_functions)
print("\nc:")
print(c.grad_fn)
print("\nb:")
print(b.grad_fn)
print("\na:")
print(a.grad_fn)

With all this machinery in place, how do we get derivatives out? You
call the `backward()` method on the output, and check the input's `grad`
property to inspect the gradients:  
有了这套机制，我们该如何获取导数呢？你只需在输出张量上调用 `backward()` 方法，然后检查输入张量的 `grad` 属性即可查看梯度：

In [ ]:
out.backward()
print(a.grad)
plt.plot(a.detach(), a.grad.detach())

这个报错 `RuntimeError: grad can be implicitly created only for scalar outputs` 是 PyTorch 中非常经典的一个错误。

### 为什么会报错？
在 PyTorch 中，`backward()` 方法默认**只支持对标量（0维张量，即一个单独的数值）进行反向传播**。

让我们看看你的代码：
1. `a` 是一个包含 25 个元素的张量（形状为 `[25]`）。
2. 经过 `sin`、乘法、加法后，`d` 和 `out` 依然是包含 25 个元素的向量（形状为 `[25]`）。
3. 当你直接对 `out`（一个向量）调用 `out.backward()` 时，PyTorch 就懵了：向量对参数求导得到的是一个雅可比矩阵（Jacobian Matrix），而不是一个简单的梯度向量，优化器不知道该如何处理这个矩阵。因此，它要求最终的输出必须是一个标量（Scalar）。

### 如何解决？
你需要将这 25 个值“聚合”成一个单独的数值（标量）。最常用、最推荐的做法就是求和（`.sum()`）或求平均值（`.mean()`）。

**修改后的正确代码：**
```python
import torch
import math

a = torch.linspace(0.0, 2.0 * math.pi, steps=25, requires_grad=True) 
b = torch.sin(a)
c = 2 * b
d = c + 1

# ✅ 正确做法：使用 .sum() 将向量转换为标量（0维张量）
out = d.sum()  

# 现在 out 是一个单独的数值，可以成功反向传播了
out.backward() 

# 查看输入 a 的梯度
print(a.grad)
```

### 💡 补充说明：
其实你在代码里已经写出了正确的做法（`out = d.sum()`），只是被注释掉了。在实际的深度学习训练中，损失函数（Loss）通常都是对一批（Batch）样本的误差进行求和或求平均，最终变成一个标量，然后再调用 `loss.backward()` 的。

在 PyTorch 中，`tensor.grad` 和 `tensor.grad_fn` 是自动求导机制（Autograd）中两个完全不同但又紧密相关的属性。简单来说，**`grad_fn` 是记录计算过程的“导航器”，而 `grad` 是最终计算出来的“梯度结果”**。

我们可以从以下几个方面来区分它们：

### 1. 核心含义的区别
*   **`grad_fn`（梯度函数）**：它用来**记录变量是怎么计算出来的**，方便后续计算梯度。它是反向传播的起点和导航器，当你对一个张量执行可微操作时，PyTorch 会创建一个对应的 `Function` 对象，而 `grad_fn` 就指向这个对象。
*   **`grad`（梯度值）**：它是**张量本身的梯度结果**。当执行完 `backward()`（反向传播）之后，计算出的梯度值就会自动累加并存储在这个属性中。

### 2. 产生与赋值的时机
*   **`grad_fn`**：在**前向传播（计算过程）**中自动产生。只要你对一个张量进行了数学运算（如加减乘除），生成的新张量就会带有 `grad_fn`。
*   **`grad`**：在**反向传播（调用 `backward()`）**之后才会被赋值。在反向传播之前，它的值通常是 `None`。

### 3. 适用对象的区别（重点）
*   **`grad_fn`**：通常属于**中间结果张量**（非叶子节点）。因为它是通过运算产生的。如果是用户直接创建的张量（叶子节点），它的 `grad_fn` 是 `None`。
*   **`grad`**：通常存储在**叶子节点张量**（如模型的权重参数、输入数据）上。这些是我们最终需要更新或查看梯度的变量。

### 4. 代码直观对比
我们来看一个具体的例子：

```python
import torch

# 1. 创建一个叶子节点张量（直接创建，没有 grad_fn）
x = torch.tensor([2.0], requires_grad=True)
print(x.grad_fn)  # 输出: None
print(x.grad)     # 输出: None

# 2. 进行一次运算（产生中间结果 y，y 有 grad_fn）
y = x ** 2
print(y.grad_fn)  # 输出: <PowBackward0 object> (记录了 y 是通过平方运算来的)

# 3. 进行反向传播
y.backward()

# 4. 查看叶子节点 x 的梯度
print(x.grad)     # 输出: tensor([4.]) (因为 dy/dx = 2x, 2*2=4)
```

### 总结
你可以把 `grad_fn` 想象成**“菜谱”**，它记录了这道菜（张量）是用什么食材、经过什么步骤做出来的；而 `grad` 则是最终算出来的**“调料配比（梯度）”**，它告诉你在训练模型时，参数应该往哪个方向、以多大的幅度去更新。

Recall the computation steps we took to get here:  
回顾我们在此之前的计算步骤：

``` {.python}
a = torch.linspace(0., 2. * math.pi, steps=25, requires_grad=True)
b = torch.sin(a)
c = 2 * b
d = c + 1
out = d.sum()
```

Adding a constant, as we did to compute `d`, does not change the
derivative. That leaves $c = 2 * b = 2 * \sin(a)$, the derivative of
which should be $2 * \cos(a)$. Looking at the graph above, that's just
what we see.  
正如我们在计算 `d` 时所做的那样，加上一个常数并不会改变导数。接下来看 $c = 2 * b = 2 * \sin(a)$，它的导数应该是 $2 * \cos(a)$。观察上面的计算图，这正是我们所看到的结果。

Be aware that only *leaf nodes* of the computation have their gradients
computed. If you tried, for example, `print(c.grad)` you'd get back
`None`. In this simple example, only the input is a leaf node, so only
it has gradients computed.  
需要注意的是，只有计算图中的**叶子节点**（leaf nodes）才会计算并保留梯度。例如，如果你尝试执行 `print(c.grad)`，将会返回 `None`。在这个简单的例子中，只有输入变量是叶子节点，因此只有它会被计算并保存梯度。

Autograd in Training  
训练中的自动求导（Autograd）
====================

We've had a brief look at how autograd works, but how does it look when
it's used for its intended purpose? Let's define a small model and
examine how it changes after a single training batch. First, define a
few constants, our model, and some stand-ins for inputs and outputs:  
我们已经简单了解了自动求导（autograd）的工作原理，但在其实际应用场景中它又是怎样的呢？让我们来定义一个小型模型，并观察它在经过单个训练批次（batch）后是如何变化的。首先，定义几个常量、我们的模型，以及用于替代输入和输出的变量：



In [ ]:
BATCH_SIZE = 16
DIM_IN = 1000
HIDDEN_SIZE = 100
DIM_OUT = 10


class TinyModel(torch.nn.Module):

    def __init__(self):
        super(TinyModel, self).__init__()

        self.layer1 = torch.nn.Linear(DIM_IN, HIDDEN_SIZE)
        self.relu = torch.nn.ReLU()
        self.layer2 = torch.nn.Linear(HIDDEN_SIZE, DIM_OUT)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x


some_input = torch.randn(BATCH_SIZE, DIM_IN, requires_grad=False)
ideal_output = torch.randn(BATCH_SIZE, DIM_OUT, requires_grad=False)

model = TinyModel()

One thing you might notice is that we never specify `requires_grad=True`
for the model's layers. Within a subclass of `torch.nn.Module`, it's
assumed that we want to track gradients on the layers' weights for
learning.  
你可能会注意到，我们从未显式地为模型的层指定 requires_grad=True。在 torch.nn.Module 的子类中，系统会默认我们需要追踪层权重的梯度以用于学习。

If we look at the layers of the model, we can examine the values of the
weights, and verify that no gradients have been computed yet:  
如果我们查看模型的各个层，可以检查权重的值，并验证此时尚未计算任何梯度：


In [ ]:
print(model.layer2.weight[0][0:10])  # just a small slice
print(model.layer2.weight.grad)

Let's see how this changes when we run through one training batch. For a
loss function, we'll just use the square of the Euclidean distance
between our `prediction` and the `ideal_output`, and we'll use a basic
stochastic gradient descent optimizer.  
让我们看看在运行一个训练批次后会发生什么变化。对于损失函数，我们直接使用 prediction（预测值）和 ideal_output（理想输出）之间欧氏距离的平方，并使用基础随机梯度下降（SGD）优化器。


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

prediction = model(some_input)

loss = (ideal_output - prediction).pow(2).sum()
print(loss)

Now, let's call `loss.backward()` and see what happens:  
现在，让我们调用 loss.backward() 来看看会发生什么：


In [ ]:
loss.backward()
print(model.layer2.weight[0][0:10])
print(model.layer2.weight.grad[0][0:10])

We can see that the gradients have been computed for each learning
weight, but the weights remain unchanged, because we haven't run the
optimizer yet. The optimizer is responsible for updating model weights
based on the computed gradients.  
我们可以看到，每个可学习权重都已经计算出了梯度，但权重本身并未改变，因为我们还没有运行优化器。优化器负责根据计算出的梯度来更新模型权重。


In [ ]:
optimizer.step()
print(model.layer2.weight[0][0:10])
print(model.layer2.weight.grad[0][0:10])

You should see that `layer2`'s weights have changed.  
此时你应该能看到 layer2 的权重已经发生了变化。

One important thing about the process: After calling `optimizer.step()`,
you need to call `optimizer.zero_grad()`, or else every time you run
`loss.backward()`, the gradients on the learning weights will
accumulate:  
关于该过程有一点非常重要：在调用 `optimizer.step()` 之后，你必须调用 `optimizer.zero_grad()`，否则每次运行 `loss.backward()` 时，可学习权重的梯度都会不断累积：

- 因为.grad是累加的，为了实现小内存，逐渐增加到大批次，这是原始设计功能，所以要清零 https://www.qianwen.com/quarkshare/apps/mshare/routes/share-v3?biz_id=ai_chat_v2&env=prod&share_id=c14d41206e7e42c79dfe30d1b6c045d4&source_client=pc&feature_chat_domain=https%3A%2F%2Fchat-api-quark2.qianwen.com

In [ ]:
print(model.layer2.weight.grad[0][0:10])

for i in range(0, 5):
    prediction = model(some_input)
    loss = (ideal_output - prediction).pow(2).sum()
    loss.backward()

print(model.layer2.weight.grad[0][0:10])  # 错误例子 不清零

optimizer.zero_grad(set_to_none=False)

print(model.layer2.weight.grad[0][0:10])

After running the cell above, you should see that after running
`loss.backward()` multiple times, the magnitudes of most of the
gradients will be much larger. Failing to zero the gradients before
running your next training batch will cause the gradients to blow up in
this manner, causing incorrect and unpredictable learning results.  
运行完上面的代码单元后，你应该会看到，在多次运行 `loss.backward()` 后，大部分梯度的数值会变得大得多。如果在运行下一个训练批次之前未能将梯度清零，梯度就会以这种方式爆炸式增长，从而导致错误且不可预测的学习结果。

Turning Autograd Off and On  
开启与关闭自动求导（Autograd）
===========================

There are situations where you will need fine-grained control over
whether autograd is enabled. There are multiple ways to do this,
depending on the situation.  
在某些情况下，你需要对是否启用自动求导进行细粒度的控制。根据具体情况的不同，有多种方法可以实现这一点。

The simplest is to change the `requires_grad` flag on a tensor directly:  
最简单的方法是直接更改张量的 `requires_grad` 标志：

In [ ]:
a = torch.ones(2, 3, requires_grad=True)
print(a)

b1 = 2 * a # 有自动求导
print(b1)

a.requires_grad = False # 关闭自动求导
b2 = 2 * a
print(b2)

In the cell above, we see that `b1` has a `grad_fn` (i.e., a traced
computation history), which is what we expect, since it was derived from
a tensor, `a`, that had autograd turned on. When we turn off autograd
explicitly with `a.requires_grad = False`, computation history is no
longer tracked, as we see when we compute `b2`.  
在上面的代码单元中，我们看到 `b1` 拥有一个 `grad_fn`（即被追踪的计算历史），这是符合预期的，因为它是由开启了自动求导的张量 `a` 衍生而来的。当我们通过 `a.requires_grad = False` 显式关闭自动求导后，计算历史将不再被追踪，正如我们在计算 `b2` 时所看到的那样。

If you only need autograd turned off temporarily, a better way is to use
the `torch.no_grad()`:  
如果你只需要临时关闭自动求导，更好的方法是使用 `torch.no_grad()`：



In [ ]:
a = torch.ones(2, 3, requires_grad=True) * 2
b = torch.ones(2, 3, requires_grad=True) * 3

c1 = a + b
print(c1)  # 有自动求导

with torch.no_grad(): # 临时关闭
    c2 = a + b

print(c2) # 没有自动求导

c3 = a * b # 有自动求导，脱离with操作，还有
print(c3)

`torch.no_grad()` can also be used as a function or method decorator:  
`torch.no_grad()` 也可以作为函数或方法的装饰器来使用：  


https://www.qianwen.com/quarkchat/uoVJlYzEfEkby0cGSX0eoArkYsuxwS21?entry=homepage&entry_l2=bar_switch_active
关闭自动求导，减少内存，在测试使用



In [ ]:
def add_tensors1(x, y):
    return x + y


@torch.no_grad() #装饰器
def add_tensors2(x, y):
    return x + y


a = torch.ones(2, 3, requires_grad=True) * 2
b = torch.ones(2, 3, requires_grad=True) * 3

c1 = add_tensors1(a, b) # 有自动求导函数
print(c1)

c2 = add_tensors2(a, b) # 无自动求导函数
print(c2)

There's a corresponding context manager, `torch.enable_grad()`, for
turning autograd on when it isn't already. It may also be used as a
decorator.  
还有一个对应的上下文管理器 `torch.enable_grad()`，用于在自动求导未开启时将其打开。它同样可以作为装饰器使用。

Finally, you may have a tensor that requires gradient tracking, but you
want a copy that does not. For this we have the `Tensor` object's
`detach()` method - it creates a copy of the tensor that is *detached*
from the computation history:  
最后，你可能有一个需要追踪梯度的张量，但同时又想要一个不需要追踪梯度的副本。为此，我们可以使用 `Tensor` 对象的 `detach()` 方法——它会创建一个与计算历史*分离*的张量副本：

In [ ]:
x = torch.rand(5, requires_grad=True) # 有梯度追踪
y = x.detach() # 无梯度最终。matplotlib需要numpy数组

print(x)
print(y)

We did this above when we wanted to graph some of our tensors. This is
because `matplotlib` expects a NumPy array as input, and the implicit
conversion from a PyTorch tensor to a NumPy array is not enabled for
tensors with requires\_grad=True. Making a detached copy lets us move
forward.  
我们在前面想要绘制张量图表时就使用了这个方法。这是因为 `matplotlib` 期望输入的是 NumPy 数组，而对于 `requires_grad=True` 的张量，PyTorch 默认不允许将其隐式转换为 NumPy 数组。创建一个分离的副本（detached copy）就能让我们继续进行操作。

Autograd and In-place Operations  
自动求导与原地操作（In-place Operations）
================================

In every example in this notebook so far, we've used variables to
capture the intermediate values of a computation. Autograd needs these
intermediate values to perform gradient computations. *For this reason,
you must be careful about using in-place operations when using
autograd.* Doing so can destroy information you need to compute
derivatives in the `backward()` call. PyTorch will even stop you if you
attempt an in-place operation on leaf variable that requires autograd,
as shown below.  
到目前为止，本笔记中的所有示例都使用了变量来保存计算的中间值。自动求导（Autograd）需要这些中间值来执行梯度计算。*因此，在使用自动求导时，必须谨慎使用原地操作（in-place operations）。* 这样做会破坏在调用 `backward()` 计算导数时所需的信息。如果你尝试对需要自动求导的叶子变量执行原地操作，PyTorch 甚至会直接阻止你，如下所示。



<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>The following code cell throws a runtime error. This is expected.下面的代码单元会抛出一个运行时错误，这是符合预期的。<pre><code>a = torch.linspace(0., 2. * math.pi, steps=25, requires_grad=True)
a.sin_()</code></pre></p>

</div>



In [ ]:
a = torch.linspace(0, 2. * math.pi, steps=25, requires_grad=True)
a.sin_() # 原地修改 有梯度追踪 报错

Autograd Profiler  
Autograd 分析器
=================

Autograd tracks every step of your computation in detail. Such a
computation history, combined with timing information, would make a
handy profiler - and autograd has that feature baked in. Here's a quick
example usage:  
Autograd 会详细记录你计算的每一步。这种计算历史记录结合时间信息，可以成为一个非常实用的性能分析器——而 autograd 已经内置了这一功能。下面是一个简单的使用示例：


In [38]:
device = torch.device("cpu")
run_on_gpu = False
if torch.cuda.is_available():
    device = torch.device("cuda")
    run_on_gpu = True

print(run_on_gpu)
x = torch.randn(2, 3, requires_grad=True)
y = torch.rand(2, 3, requires_grad=True)
z = torch.ones(2, 3, requires_grad=True)

with torch.autograd.profiler.profile(use_cuda=run_on_gpu) as prf:
    for _ in range(1000):
        z = (z / x) * y

print(prf.key_averages().table(sort_by="self_cpu_time_total"))

False
-------------  ------------  ------------  ------------  ------------  ------------  ------------  
         Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------  ------------  ------------  ------------  ------------  ------------  ------------  
    aten::div        53.55%       1.676ms        53.55%       1.676ms       1.676us          1000  
    aten::mul        46.45%       1.454ms        46.45%       1.454ms       1.454us          1000  
-------------  ------------  ------------  ------------  ------------  ------------  ------------  
Self CPU time total: 3.131ms



USDT:2026-06-24 19:19:05 35021:2086629 ActivityProfilerController.cpp:415] profiler_start
USDT:2026-06-24 19:19:05 35021:2086629 ActivityProfilerController.cpp:455] profiler_stop


In [39]:
device = torch.device("cpu")
run_on_gpu = False
if torch.cuda.is_available():
    device = torch.device("cuda")
    run_on_gpu = True

print(run_on_gpu)
x = torch.randn(2, 3, requires_grad=True)
y = torch.rand(2, 3, requires_grad=True)
z = torch.ones(2, 3, requires_grad=True)

with torch.autograd.profiler.profile(use_cuda=run_on_gpu) as prf:
    z = (z / x) * y

print(prf.key_averages().table(sort_by="self_cpu_time_total"))

False
-------------  ------------  ------------  ------------  ------------  ------------  ------------  
         Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------  ------------  ------------  ------------  ------------  ------------  ------------  
    aten::div        81.13%       1.618ms        81.13%       1.618ms       1.618ms             1  
    aten::mul        18.87%     376.452us        18.87%     376.452us     376.452us             1  
-------------  ------------  ------------  ------------  ------------  ------------  ------------  
Self CPU time total: 1.995ms



USDT:2026-06-24 19:21:17 35021:2086629 ActivityProfilerController.cpp:415] profiler_start
USDT:2026-06-24 19:21:17 35021:2086629 ActivityProfilerController.cpp:455] profiler_stop


The profiler can also label individual sub-blocks of code, break out the
data by input tensor shape, and export data as a Chrome tracing tools
file. For full details of the API, see the
[documentation](https://pytorch.org/docs/stable/autograd.html#profiler).  
性能分析器还能够标记代码中的独立子代码块，按输入张量的形状对数据进行分类统计，并将数据导出为 Chrome tracing tools 工具的文件格式。有关 API 的完整详细信息，请参阅[文档](https://pytorch.org/docs/stable/autograd.html#profiler)。

Advanced Topic: More Autograd Detail and the High-Level API  
高级主题：更详细的 Autograd 原理与高级 API
===========================================================

If you have a function with an n-dimensional input and m-dimensional
output, $\vec{y}=f(\vec{x})$, the complete gradient is a matrix of the
derivative of every output with respect to every input, called the
*Jacobian:*  
如果你有一个 n 维输入和 m 维输出的函数 $\vec{y}=f(\vec{x})$，其完整的梯度就是每一个输出对每一个输入的导数矩阵，该矩阵被称为**雅可比矩阵 (Jacobian)**：

$$\begin{aligned}
J
=
\left(\begin{array}{ccc}
\frac{\partial y_{1}}{\partial x_{1}} & \cdots & \frac{\partial y_{1}}{\partial x_{n}}\\
\vdots & \ddots & \vdots\\
\frac{\partial y_{m}}{\partial x_{1}} & \cdots & \frac{\partial y_{m}}{\partial x_{n}}
\end{array}\right)
\end{aligned}$$

If you have a second function, $l=g\left(\vec{y}\right)$ that takes
m-dimensional input (that is, the same dimensionality as the output
above), and returns a scalar output, you can express its gradients with
respect to $\vec{y}$ as a column vector,  
$v=\left(\begin{array}{ccc}\frac{\partial l}{\partial y_{1}} & \cdots & \frac{\partial l}{\partial y_{m}}\end{array}\right)^{T}$  
如果你有第二个函数 $l=g\left(\vec{y}\right)$，它接受 m 维输入（即与上述输出维度相同），并返回一个标量输出，你可以将其关于 $\vec{y}$ 的梯度表示为一个列向量，
$v=\left(\begin{array}{ccc}\frac{\partial l}{\partial y_{1}} & \cdots & \frac{\partial l}{\partial y_{m}}\end{array}\right)^{T}$
- which is really just a one-column Jacobian.  
—— 这实际上就是一个只有一列的雅可比矩阵。

More concretely, imagine the first function as your PyTorch model (with
potentially many inputs and many outputs) and the second function as a
loss function (with the model's output as input, and the loss value as
the scalar output).  
更具体地说，想象第一个函数是你的 PyTorch 模型（可能有多个输入和多个输出），第二个函数是损失函数（以模型的输出作为输入，以损失值作为标量输出）。


If we multiply the first function's Jacobian by the gradient of the
second function, and apply the chain rule, we get:  
如果我们把第一个函数的雅可比矩阵乘以第二个函数的梯度，并应用链式法则，就会得到：

$$\begin{aligned}
J^{T}\cdot v=\left(\begin{array}{ccc}
\frac{\partial y_{1}}{\partial x_{1}} & \cdots & \frac{\partial y_{m}}{\partial x_{1}}\\
\vdots & \ddots & \vdots\\
\frac{\partial y_{1}}{\partial x_{n}} & \cdots & \frac{\partial y_{m}}{\partial x_{n}}
\end{array}\right)\left(\begin{array}{c}
\frac{\partial l}{\partial y_{1}}\\
\vdots\\
\frac{\partial l}{\partial y_{m}}
\end{array}\right)=\left(\begin{array}{c}
\frac{\partial l}{\partial x_{1}}\\
\vdots\\
\frac{\partial l}{\partial x_{n}}
\end{array}\right)
\end{aligned}$$

Note: You could also use the equivalent operation $v^{T}\cdot J$, and
get back a row vector.  
注意：你也可以使用等效运算 $v^{T}\cdot J$，结果会是一个行向量。

The resulting column vector is the *gradient of the second function with
respect to the inputs of the first* - or in the case of our model and
loss function, the gradient of the loss with respect to the model
inputs.  
所得的列向量就是**第二个函数关于第一个函数输入的梯度** —— 或者在我们的模型和损失函数的例子中，即为损失函数关于模型输入的梯度。

**\`\`torch.autograd\`\` is an engine for computing these products.**
This is how we accumulate the gradients over the learning weights during
the backward pass.  
**\`\`torch.autograd\`\` 就是用于计算这些乘积的引擎。**
这正是我们在反向传播过程中累积学习权重梯度的方式。

For this reason, the `backward()` call can *also* take an optional
vector input. This vector represents a set of gradients over the tensor,
which are multiplied by the Jacobian of the autograd-traced tensor that
precedes it. Let's try a specific example with a small vector:  
正因为如此，`backward()` 调用也可以接收一个可选的向量输入。这个向量代表了张量上的一组梯度，它们将与前置的 autograd 追踪张量的雅可比矩阵相乘。让我们尝试一个带有小向量的具体例子：


In [49]:
import torch
x = torch.randn(3, requires_grad=True)

y = x * 2
while y.data.norm() < 1000:
    y = y * 2

print(y)

tensor([-926.2525,  396.9276, -359.9319], grad_fn=<MulBackward0>)


### 3. 具体计算过程
结合你的代码 `x.data.norm()`，它的底层计算逻辑如下：
1. 取出张量 `x` 中的每一个元素。
2. 对每个元素进行**平方**。
3. 将所有平方后的值**求和**。
4. 对求和的结果**开平方根**。


在数学和机器学习中，**范数（Norm）**通俗地说，就是用来衡量一个向量（或矩阵）“大小”、“长度”或“绝对值”的度量标准。

你可以把它理解为我们在中学学过的**绝对值**在多维空间中的升级版。

### 1. 直观理解：从一维到多维
*   **一维空间（数轴）**：衡量一个数 $x$ 离原点的距离，我们用**绝对值** $|x|$。
*   **二维空间（平面）**：衡量一个点 $(x, y)$ 离原点的距离，我们用勾股定理 $\sqrt{x^2 + y^2}$。
*   **多维空间（张量/向量）**：衡量一个包含多个元素的张量离原点的距离，我们就用**范数**。

### 2. PyTorch 中的默认范数：L2 范数
在 PyTorch 中，当你直接调用 `.norm()` 时，默认计算的是 **L2 范数**（也叫欧几里得范数）。

它的计算公式是：**将向量中所有元素的平方相加，然后再开平方根。**

**举个具体的例子：**
假设有一个一维张量（向量） $x = [3, 4]$
*   计算 L2 范数：$\sqrt{3^2 + 4^2} = \sqrt{9 + 16} = \sqrt{25} = 5$
*   这个 `5` 就是该张量的 L2 范数，它代表了向量 $[3, 4]$ 的长度。

### 3. 为什么在深度学习里经常用范数？
结合你之前问的 `while y.data.norm() < 1000:` 代码，范数在深度学习中有几个非常重要的作用：

1.  **衡量模型权重的大小（防止过拟合）**：
    在训练神经网络时，如果权重矩阵的范数变得非常大，说明模型可能“学偏了”或者发生了过拟合。我们通常会加入“正则化（Regularization）”项（如 L2 正则化），其本质就是**惩罚过大的范数**，强迫模型保持较小的权重。
2.  **控制循环或梯度爆炸**：
    就像你代码里写的，用范数来判断当前张量的值是否已经膨胀到了危险的程度（比如大于 1000）。如果范数过大，通常意味着发生了“梯度爆炸”或数值溢出，此时需要停止计算或进行截断。
3.  **衡量误差（Loss）**：
    我们常说的“均方误差（MSE）”，本质上也就是预测值和真实值之间差值的 L2 范数的平方。

### 4. 其他常见的范数
除了默认的 L2 范数，你可能还会遇到：
*   **L1 范数**：所有元素绝对值之和（$\sum |x_i|$）。常用于让模型产生稀疏性（即让很多权重变成 0）。
*   **L0 范数**：向量中非零元素的个数。
*   **L∞ 范数**：向量中绝对值最大的那个元素的值。

**总结：** 当你看到 `.norm()` 时，你只需要在脑海中把它翻译成**“计算这个张量所有元素综合起来的绝对大小”**就可以了。

If we tried to call `y.backward()` now, we'd get a runtime error and a
message that gradients can only be *implicitly* computed for scalar
outputs. For a multi-dimensional output, autograd expects us to provide
gradients for those three outputs that it can multiply into the
Jacobian:  
如果此时我们尝试调用 `y.backward()`，将会遇到运行时错误，并提示只能对**标量输出**进行*隐式*梯度计算。对于多维输出，autograd 期望我们提供这三个输出的梯度，以便将其与雅可比矩阵相乘：

In [ ]:
v = torch.tensor([0.1, 1.0, 0.0001], dtype=torch.float)  # stand-in for gradients
y.backward(v)

print(x.grad)

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

非常抱歉，我之前的解释确实没有完全贴合你提供的具体数据。现在，我们用你**真实的输出数据**，把这段代码的底层逻辑一步步、严丝合缝地推导出来。

### 第一步：前向传播（计算最终的 y）
你的初始输入是：
`x = [-2.5707, 1.5935, 0.7964]`

代码首先执行了一次 `y = x * 2`，然后进入 `while` 循环不断翻倍。
根据你输出的 `y` 的值：
`y = [-1316.2155, 815.8766, 407.7587]`

我们拿第一个元素算一下：`-2.5707 * 512 = -1316.1984`（与 -1316.2155 非常接近，微小差异来自浮点数精度）。
**结论**：这说明 `while` 循环一共执行了 **8 次**。
因为 $2^9 = 512$（初始乘 2 算 1 次，循环 8 次，总共乘了 9 个 2）。
所以，整个前向传播的数学公式等价于：**`y = x * 512`**。

---

### 第二步：反向传播（计算梯度）
在 PyTorch 中，当输出 `y` 是一个向量时，`y.backward(v)` 的底层数学原理是**向量-雅可比乘积（VJP）**。

1. **计算雅可比矩阵（Jacobian）**：
   因为 `y = x * 512`，这是一个简单的线性缩放。`y` 对 `x` 求导，结果就是一个对角线全为 512 的矩阵：
   ```text
   J = [ 512,    0,    0 ]
       [   0,  512,    0 ]
       [   0,    0,  512 ]
   ```
   *(这就是为什么梯度里必然包含 512 这个“2 的幂”)*

2. **执行 `y.backward(v)`**：
   你传入的外部梯度向量 `v` 是：
   `v = [0.1, 1.0, 0.0001]`

   PyTorch 会将 `v` 与雅可比矩阵 `J` 相乘（对应元素相乘，即点积）：
   *   `x.grad[0]` = `v[0] * J[0][0]` = $0.1 \times 512 = \mathbf{51.2}$
   *   `x.grad[1]` = `v[1] * J[1][1]` = $1.0 \times 512 = \mathbf{512.0}$
   *   `x.grad[2]` = `v[2] * J[2][2]` = $0.0001 \times 512 = \mathbf{0.0512}$

---

### 第三步：验证你的输出
我们来看你打印出的最后一个结果：
`tensor([5.1200e+01, 5.1200e+02, 5.1200e-02])`

把它转换成普通十进制：
*   `5.1200e+01` = **51.2**
*   `5.1200e+02` = **512.0**
*   `5.1200e-02` = **0.0512**

**完美吻合！**

### 💡 总结你的疑惑：什么是向量的梯度更新？
在标量（单个数字）求导时，梯度就是一个数字。但在**向量求导**时：
1. 前向传播中，输入 `x` 的每个元素独立地进行了相同的数学运算（乘了 512）。
2. 反向传播时，`y.backward(v)` 并不是让向量里的元素互相求导，而是**把外部传入的梯度 `v` 当作“权重”**，分别乘以对应位置的局部导数（512）。
3. 最终得到的 `x.grad`，就是 `v` 和 `512` 逐元素相乘的结果。这就是为什么输出的梯度恰好是 `51.2`, `512.0`, `0.0512`。

---
要不要我用一张图把 VJP（向量-雅可比乘积）的几何意义画出来？直观感受一下为什么是"对应元素相乘"。

(Note that the output gradients are all related to powers of two - which
we'd expect from a repeated doubling operation.)  
（请注意，输出的梯度都是 2 的幂——这正是我们预期在连续翻倍操作中会出现的结果。）

The High-Level API  
高级 API
==================

There is an API on autograd that gives you direct access to important
differential matrix and vector operations. In particular, it allows you
to calculate the Jacobian and the *Hessian* matrices of a particular
function for particular inputs. (The Hessian is like the Jacobian, but
expresses all partial *second* derivatives.) It also provides methods
for taking vector products with these matrices.  
autograd 提供了一个 API，可以直接访问重要的微分矩阵和向量运算。特别是，它允许你计算特定函数在特定输入下的雅可比矩阵和**海森 (Hessian)** 矩阵。（海森矩阵类似于雅可比矩阵，但它表示所有的偏*二阶*导数。）该 API 还提供了对这些矩阵进行向量乘积运算的方法。

Let's take the Jacobian of a simple function, evaluated for a 2
single-element inputs:  
让我们计算一个简单函数的雅可比矩阵，该函数在 2 个单元素输入上进行求值：


In [51]:
def exp_adder(x, y):
    return 2 * x.exp() + 3 * y


inputs = (torch.rand(1), torch.rand(1))  # arguments for the function
print(inputs)
torch.autograd.functional.jacobian(exp_adder, inputs)

(tensor([0.0633]), tensor([0.4082]))


(tensor([[2.1306]]), tensor([[3.]]))

If you look closely, the first output should equal $2e^x$ (since the
derivative of $e^x$ is $e^x$), and the second value should be 3.  
仔细观察，第一个输出应该等于 $2e^x$（因为 $e^x$ 的导数是 $e^x$），第二个值应该是 3。

You can, of course, do this with higher-order tensors:  
当然，你也可以对高阶张量执行此操作：


In [52]:
inputs = (torch.rand(3), torch.rand(3))  # arguments for the function
print(inputs)
torch.autograd.functional.jacobian(exp_adder, inputs)

(tensor([0.4058, 0.7092, 0.5701]), tensor([0.6088, 0.4828, 0.0227]))


(tensor([[3.0010, 0.0000, 0.0000],
         [0.0000, 4.0646, 0.0000],
         [0.0000, 0.0000, 3.5368]]),
 tensor([[3., 0., 0.],
         [0., 3., 0.],
         [0., 0., 3.]]))

The `torch.autograd.functional.hessian()` method works identically
(assuming your function is twice differentiable), but returns a matrix
of all second derivatives.  
`torch.autograd.functional.hessian()` 方法的工作原理完全相同（假设你的函数是二阶可微的），但它返回的是一个包含所有二阶导数的矩阵。

There is also a function to directly compute the vector-Jacobian
product, if you provide the vector:  
此外，如果你提供向量，也可以使用函数直接计算**向量-雅可比积（vector-Jacobian product）**：

In [ ]:
def do_some_doubling(x):
    y = x * 2
    while y.data.norm() < 1000:
        y = y * 2
    return y


inputs = torch.randn(3)
my_gradients = torch.tensor([0.1, 1.0, 0.0001])
torch.autograd.functional.vjp(do_some_doubling, inputs, v=my_gradients) # 提供函数

(tensor([ -431.4142,  -314.5630, -1087.4307]),
 tensor([5.1200e+01, 5.1200e+02, 5.1200e-02]))

In [55]:
import torch
x = torch.randn(3, requires_grad=True)

y = x * 2
while y.data.norm() < 1000:
    y = y * 2

v = torch.tensor([0.1, 1.0, 0.0001], dtype=torch.float)  # stand-in for gradients
y.backward(v)

print(x.grad)

tensor([1.0240e+02, 1.0240e+03, 1.0240e-01])


The `torch.autograd.functional.jvp()` method performs the same matrix
multiplication as `vjp()` with the operands reversed. The `vhp()` and
`hvp()` methods do the same for a vector-Hessian product.  
`torch.autograd.functional.jvp()` 方法执行与 `vjp()` 相同的矩阵乘法，只是操作数的顺序相反。`vhp()` 和 `hvp()` 方法则用于计算**向量-海森积（vector-Hessian product）**。

For more information, including performance notes on the [docs for the
functional
API](https://pytorch.org/docs/stable/autograd.html#functional-higher-level-api)  
有关更多信息（包括功能 API 的性能说明），请参阅[官方文档](https://pytorch.org/docs/stable/autograd.html#functional-higher-level-api)。


这两个代码块在**数学本质上是完全等价的**（它们底层调用的都是反向传播算法），但在**工程实现、代码风格和适用场景**上有显著的区别。

### 1. 两者的核心区别

| 对比维度 | 代码块 2：`y.backward(v)` (传统方式) | 代码块 1：`torch.autograd.functional.vjp` (函数式方式) |
| :--- | :--- | :--- |
| **编程范式** | **面向对象 (OOP)**：依赖张量对象的内部状态。 | **函数式编程 (Functional)**：纯函数，无副作用。 |
| **状态管理** | 需要手动创建 `requires_grad=True`，计算后梯度会**累积**在 `x.grad` 中，下次计算前必须手动 `x.grad.zero_()`。 | 不需要 `requires_grad=True`。每次调用都是独立的，**不会**产生状态累积，无需手动清零。 |
| **返回值** | 无返回值。结果直接修改输入张量的 `.grad` 属性。 | 返回一个元组 `(output, vjp_result)`，既包含前向传播的结果，也包含反向传播的梯度。 |
| **适用场景** | 深度学习模型训练（如 `loss.backward()`）。 | 高级研究、算法验证、需要同时获取前向输出和梯度的场景。 |

---

### 2. `torch.autograd.functional.vjp` 有什么功能？

`vjp` 是 **Vector-Jacobian Product（向量-雅可比乘积）** 的缩写。它的核心功能就是**以纯函数的形式，一次性完成“前向计算 + 反向求导”的完整过程**。

#### 它的标准用法：
```python
# 1. 定义一个纯计算函数（不需要 requires_grad）
def do_some_doubling(x):
    y = x * 2
    while y.data.norm() < 1000:
        y = y * 2
    return y

# 2. 准备输入和外部梯度
inputs = torch.randn(3)
my_gradients = torch.tensor([0.1, 1.0, 0.0001])

# 3. 调用 vjp
output, vjp_result = torch.autograd.functional.vjp(do_some_doubling, inputs, v=my_gradients)
```

#### 它的三大优势：
1. **安全且无状态**：在代码块 2 中，如果你连续调用两次 `y.backward(v)`，第二次得到的 `x.grad` 会是第一次的两倍（因为 PyTorch 默认累加梯度）。而 `vjp` 每次都是干净的独立计算，非常适合用来做单元测试或数学验证。
2. **同时获取输出和梯度**：`backward()` 只能算梯度，而 `vjp` 会把前向传播的最终结果 `output` 和反向传播的梯度 `vjp_result` 一起打包返回，非常方便调试。
3. **支持动态计算图的高级操作**：在处理强化学习、元学习（Meta-Learning）或可微编程时，你经常需要在代码中嵌套计算梯度。传统的 `backward()` 很难处理这种嵌套，而 `vjp` 作为纯函数，可以像普通函数一样被随意嵌套和组合。

### 💡 总结
* 如果你是在**训练神经网络**，请使用 `y.backward(v)`。
* 如果你是在**做算法研究、写测试用例、或者需要在代码中动态提取梯度**，请使用 `torch.autograd.functional.vjp`。它更优雅、更安全、更符合现代 PyTorch 的函数式编程趋势。

---
要不要我也帮你把 `torch.autograd.functional.hessian` 的例子跑一遍？它和 jacobian 一起构成了完整的二阶导数分析工具链。